# 03 · Quick test in Colab

A small, fast run of the whole pipeline before the real training on Borah:

1. Builds a mini dataset from the chips already in Google Drive (a few chips per fire).
2. Scores two baselines: **persistence** (nothing changes) and **gradient boosting** on per-pixel features.
3. Trains a small Prithvi-EO-2.0 (100M, encoder frozen) to predict NBR 12 months ahead.
4. Makes the ground-truth vs prediction table and the first figures.

**Runtime → Change runtime type → T4 GPU** before running. It takes about 30–45 min.

## 0. Settings

In [ ]:
CHIPS = "/content/drive/MyDrive/cs497_hls_chips"          # GeoTIFFs from notebook 02
DATASET = "/content/dataset"                               # local disk is much faster than Drive
RESULTS = "/content/drive/MyDrive/cs497_results/quick"     # tables and figures are saved here
REPO_URL = "https://github.com/RobertojonesFAI/Post_Wildfire_Vegetation.git"

MAX_PER_FIRE = 4          # chips per fire in the mini dataset
BACKBONE = "prithvi_eo_v2_100_tl"
FRAMES = 4                # months the model sees
HORIZON = 12              # predict NBR this many months ahead
EPOCHS = 5
BATCH = 8
MAX_TRAIN = 1500          # samples
MAX_VAL = 400

In [ ]:
from google.colab import drive
drive.mount("/content/drive")

!git clone -q {REPO_URL} /content/repo
%cd /content/repo
!pip install -q terratorch rasterio

import sys
sys.path.insert(0, "/content/repo/src")
import os
import torch
os.makedirs(RESULTS, exist_ok=True)
print(torch.cuda.get_device_name(0) if torch.cuda.is_available() else "No GPU! Runtime -> Change runtime type -> T4 GPU")

## 1. Mini dataset

In [ ]:
!python scripts/build_dataset.py --chips "{CHIPS}" --out {DATASET} --max-per-fire {MAX_PER_FIRE} --workers 2

In [ ]:
import pandas as pd
from recovery.data import make_index, RecoveryDataset

index = make_index(DATASET, num_frames=FRAMES, horizon=HORIZON)
VAL_SPLIT = "val" if (index["split_fire"] == "val").any() else "test"
print("Validating on:", VAL_SPLIT)
index.groupby("split_fire").agg(samples=("chip_id", "size"), chips=("chip_id", "nunique"), fires=("mtbs_id", "nunique"))

## 2. Baselines

- **Persistence:** NBR in 12 months = the last clear NBR. If a model can't beat this, it hasn't learned anything.
- **Gradient boosting:** per-pixel features (recent NBR, pre/post-fire NBR, months since fire, target month, terrain, land cover).

In [ ]:
from torch.utils.data import DataLoader
from recovery.baselines import fit_gbm
from recovery.train import evaluate

tr = index[index["split_fire"] == "train"].sample(frac=1, random_state=497).head(MAX_TRAIN)
va = index[index["split_fire"] == VAL_SPLIT].sample(frac=1, random_state=497).head(MAX_VAL)
train_ds = RecoveryDataset(DATASET, tr)
val_ds = RecoveryDataset(DATASET, va)

gbm = fit_gbm(train_ds, pixels_per_sample=300, max_samples=800)
baselines = evaluate(None, DataLoader(val_ds, batch_size=BATCH), torch.device("cpu"), gbm=gbm)
baselines[baselines["group"] == "all"]

## 3. Prithvi (encoder frozen, small decoder)

In [ ]:
from recovery.train import train

model, prithvi_table, history = train(
    DATASET, RESULTS, backbone=BACKBONE, num_frames=FRAMES, horizon=HORIZON, epochs=EPOCHS,
    batch_size=BATCH, workers=2, max_train=MAX_TRAIN, max_val=MAX_VAL, val_split=VAL_SPLIT,
)
history

## 4. Ground truth vs prediction

Same validation pixels for every method. `skill_vs_persistence` > 0 means better than persistence (1 = perfect).

In [ ]:
results = pd.concat([baselines, prithvi_table[prithvi_table["method"].str.startswith("prithvi")]], ignore_index=True)
results = results.sort_values(["group", "method"]).reset_index(drop=True)
results.to_csv(f"{RESULTS}/results.csv", index=False)

nbr = results[~results["method"].str.contains("recovery")]
print("NBR 12 months ahead")
display(nbr.pivot(index="method", columns="group", values="rmse").round(3))
display(nbr.pivot(index="method", columns="group", values="skill_vs_persistence").round(3))
rec = results[results["method"].str.contains("recovery")]
print("Recovery %, mean absolute error (points)")
display(rec.pivot(index="method", columns="group", values="mae").round(1))

## 5. Figures

In [ ]:
import numpy as np
import matplotlib.pyplot as plt
from recovery.baselines import predict_gbm
from recovery.train import to_device

device = torch.device("cuda" if torch.cuda.is_available() else "cpu")
model.eval()
picks = sorted(range(len(val_ds)), key=lambda i: -int(val_ds[i]["mask"].sum()))[:3]

fig, axes = plt.subplots(len(picks), 5, figsize=(17, 3.6 * len(picks)))
for row, i in zip(np.atleast_2d(axes), picks):
    item = val_ds[i]
    batch = to_device({k: v.unsqueeze(0) for k, v in item.items()}, device)
    with torch.no_grad():
        pred = model(batch)[0].float().cpu().numpy()
    mask = item["mask"].numpy()
    show = lambda a: np.where(mask, a, np.nan)
    panels = [("NBR now", item["base"].numpy()), (f"Truth (+{HORIZON} mo)", item["target"].numpy()),
              ("Gradient boosting", predict_gbm(gbm, item)), ("Prithvi", pred),
              ("Prithvi error", pred - item["target"].numpy())]
    r = va.iloc[i]
    for ax, (title, a) in zip(row, panels):
        err = "error" in title
        im = ax.imshow(show(a), cmap="RdBu" if err else "RdYlGn", vmin=-0.4 if err else -0.3, vmax=0.4 if err else 0.7)
        ax.set_title(title, fontsize=10)
        ax.axis("off")
    row[0].text(0, -12, f"{r['name'].title()} · {r['month']} -> {r['target_month']}", fontsize=9)
    fig.colorbar(im, ax=row[-1], fraction=0.046)
fig.tight_layout()
fig.savefig(f"{RESULTS}/examples.png", dpi=150)
plt.show()

In [ ]:
chips = pd.read_csv(f"{DATASET}/chips.csv")
months = pd.read_csv(f"{DATASET}/chip_months.csv").merge(chips[["chip_id", "veg_type"]], on="chip_id")
months = months[(months["months_since_fire"] >= 0) & months["month"].str[5:7].isin(["06", "07", "08", "09"])]
months["years_since_fire"] = months["months_since_fire"] // 12

fig, ax = plt.subplots(figsize=(7, 4.5))
for veg, g in months.dropna(subset=["veg_type"]).groupby("veg_type"):
    curve = g.groupby("years_since_fire")["rec_burned"].median()
    ax.plot(curve.index, curve.values, marker="o", label=veg)
ax.axhline(80, color="gray", linestyle="--", linewidth=1)
ax.set_xlabel("Years since fire (summer months)")
ax.set_ylabel("Median recovery (%)")
ax.set_title("Post-fire recovery by vegetation type")
ax.legend()
fig.tight_layout()
fig.savefig(f"{RESULTS}/recovery_curves.png", dpi=150)
plt.show()

## Next

- If Prithvi beats both baselines here, the same code runs on Borah with every chip, the 300M/600M models, LoRA, and the weather fusion options.
- Everything from this run (tables, figures, the trained decoder) is in Drive → `cs497_results/quick`.